In [ ]:
"""
RETAIL SALES DATA MART — ETL + Data Cube + OLAP Operations
Dependencies: pandas, numpy, sqlite3
Sections:
  1. EXTRACT   - simulate raw retail transaction data (as if pulled from a POS system)
  2. TRANSFORM - clean, derive surrogate keys, build dimension + fact tables (star schema)
  3. LOAD      - load into a SQLite data mart (retail_datamart.db)
  4. CUBE      - build an in-memory OLAP cube using pandas pivot_table
  5. OLAP OPS  - Roll-up, Drill-down, Slice, Dice, Pivot demonstrated on the cube
  6. QUERIES   - equivalent SQL OLAP-style queries run against the data mart
"""

import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime, timedelta

np.random.seed(42)

# 1. EXTRACT — raw transactional data (simulating a POS export)

print("="*70)
print("STEP 1: EXTRACT")
print("="*70)

n_records = 2000

products = [
    ("P001", "Laptop", "Electronics"), ("P002", "Smartphone", "Electronics"),
    ("P003", "Headphones", "Electronics"), ("P004", "T-Shirt", "Apparel"),
    ("P005", "Jeans", "Apparel"), ("P006", "Jacket", "Apparel"),
    ("P007", "Blender", "Home & Kitchen"), ("P008", "Microwave", "Home & Kitchen"),
    ("P009", "Sofa", "Furniture"), ("P010", "Office Chair", "Furniture"),
]

stores = [
    ("S01", "Pune Camp", "Pune", "Maharashtra", "West"),
    ("S02", "Andheri", "Mumbai", "Maharashtra", "West"),
    ("S03", "Connaught Place", "Delhi", "Delhi", "North"),
    ("S04", "MG Road", "Bengaluru", "Karnataka", "South"),
    ("S05", "Salt Lake", "Kolkata", "West Bengal", "East"),
]

customer_segments = ["Retail", "Wholesale", "Online-only"]

start_date = datetime(2024, 1, 1)
raw_rows = []
for i in range(n_records):
    prod = products[np.random.randint(0, len(products))]
    store = stores[np.random.randint(0, len(stores))]
    days_offset = np.random.randint(0, 730)  # 2 years of data
    txn_date = start_date + timedelta(days=int(days_offset))
    qty = np.random.randint(1, 6)
    unit_price = {"Electronics": 500, "Apparel": 40, "Home & Kitchen": 80, "Furniture": 250}[prod[2]]
    unit_price = unit_price * np.random.uniform(0.85, 1.15)
    discount_pct = np.random.choice([0, 0, 0, 5, 10, 15])
    customer_id = f"C{np.random.randint(1, 400):04d}"
    segment = customer_segments[np.random.randint(0, len(customer_segments))]

    raw_rows.append({
        "txn_id": f"T{i:06d}",
        "txn_date": txn_date.strftime("%Y-%m-%d"),
        "product_code": prod[0], "product_name": prod[1], "category": prod[2],
        "store_code": store[0], "store_name": store[1], "city": store[2],
        "state": store[3], "region": store[4],
        "customer_id": customer_id, "segment": segment,
        "quantity": qty, "unit_price": round(unit_price, 2),
        "discount_pct": discount_pct,
    })

raw_df = pd.DataFrame(raw_rows)
print(f"Extracted {len(raw_df)} raw transaction records")
print(raw_df.head(3).to_string(index=False))

# 2. TRANSFORM — clean data, derive measures, build star schema

print("\n" + "="*70)
print("STEP 2: TRANSFORM (build star schema)")
print("="*70)

df = raw_df.copy()
df["txn_date"] = pd.to_datetime(df["txn_date"])

# Derived measures
df["gross_amount"] = df["quantity"] * df["unit_price"]
df["discount_amount"] = df["gross_amount"] * df["discount_pct"] / 100
df["net_sales"] = df["gross_amount"] - df["discount_amount"]
df["cost_price"] = df["unit_price"] * 0.65          # assume 65% cost ratio
df["profit"] = df["net_sales"] - (df["cost_price"] * df["quantity"])

# ---- DimDate ----
dim_date = df[["txn_date"]].drop_duplicates().reset_index(drop=True)
dim_date["date_key"] = dim_date.index + 1
dim_date["year"] = dim_date["txn_date"].dt.year
dim_date["quarter"] = "Q" + dim_date["txn_date"].dt.quarter.astype(str)
dim_date["month"] = dim_date["txn_date"].dt.strftime("%B")
dim_date["month_num"] = dim_date["txn_date"].dt.month
dim_date["day"] = dim_date["txn_date"].dt.day

# ---- DimProduct ----
dim_product = df[["product_code", "product_name", "category"]].drop_duplicates().reset_index(drop=True)
dim_product["product_key"] = dim_product.index + 1

# ---- DimStore ----
dim_store = df[["store_code", "store_name", "city", "state", "region"]].drop_duplicates().reset_index(drop=True)
dim_store["store_key"] = dim_store.index + 1

# ---- DimCustomer ----
dim_customer = df[["customer_id", "segment"]].drop_duplicates().reset_index(drop=True)
dim_customer["customer_key"] = dim_customer.index + 1

# ---- FactSales ----
fact = df.merge(dim_date, on="txn_date") \
         .merge(dim_product, on=["product_code", "product_name", "category"]) \
         .merge(dim_store, on=["store_code", "store_name", "city", "state", "region"]) \
         .merge(dim_customer, on=["customer_id", "segment"])

fact_sales = fact[["txn_id", "date_key", "product_key", "store_key", "customer_key",
                    "quantity", "unit_price", "gross_amount", "discount_amount",
                    "net_sales", "profit"]].reset_index(drop=True)
fact_sales["sales_key"] = fact_sales.index + 1

print(f"DimDate:     {len(dim_date)} rows")
print(f"DimProduct:  {len(dim_product)} rows")
print(f"DimStore:    {len(dim_store)} rows")
print(f"DimCustomer: {len(dim_customer)} rows")
print(f"FactSales:   {len(fact_sales)} rows")

# 3. LOAD — load star schema into SQLite data mart

print("\n" + "="*70)
print("STEP 3: LOAD (into SQLite data mart)")
print("="*70)
conn = sqlite3.connect("retail_datamart.db")

dim_date.to_sql("DimDate", conn, if_exists="replace", index=False)
dim_product.to_sql("DimProduct", conn, if_exists="replace", index=False)
dim_store.to_sql("DimStore", conn, if_exists="replace", index=False)
dim_customer.to_sql("DimCustomer", conn, if_exists="replace", index=False)
fact_sales.to_sql("FactSales", conn, if_exists="replace", index=False)

print("Loaded star schema into retail_datamart.db")
print("Tables:", [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table'").fetchall()])


# 4. BUILD THE OLAP CUBE (in-memory, using pandas)

print("\n" + "="*70)
print("STEP 4: BUILD DATA CUBE (Time x Product x Store)")
print("="*70)

cube_source = fact.merge(dim_date[["date_key"]], on="date_key") \
    if False else fact  # fact already has all dim attributes joined

cube = pd.pivot_table(
    cube_source,
    values=["net_sales", "quantity", "profit"],
    index=["region", "category"],
    columns=["year", "quarter"],
    aggfunc="sum",
    fill_value=0
)
print("3D Cube slice (Region x Category  ×  Year-Quarter), measure = net_sales")
print(cube["net_sales"].round(0))

# 5. OLAP OPERATIONS

print("\n" + "="*70)
print("STEP 5: OLAP OPERATIONS")
print("="*70)

# ---------- ROLL-UP: Day -> Month -> Quarter -> Year (climb the Time hierarchy)
print("\n--- ROLL-UP: Sales aggregated from Category+Store detail up to Region x Year ---")
rollup = cube_source.groupby(["region", "year"])["net_sales"].sum().round(0)
print(rollup)

# ---------- DRILL-DOWN: reverse of roll-up, Year -> Quarter -> Month
print("\n--- DRILL-DOWN: West region sales, Year -> drilled down to Month ---")
drilldown = cube_source[cube_source["region"] == "West"] \
    .groupby(["year", "month_num", "month"])["net_sales"].sum().round(0)
print(drilldown)

# ---------- SLICE: fix one dimension to a single value (e.g., Year = 2024)
print("\n--- SLICE: Year = 2024 (single slice of the Time dimension) ---")
slice_2024 = cube_source[cube_source["year"] == 2024] \
    .groupby(["region", "category"])["net_sales"].sum().round(0)
print(slice_2024)

# ---------- DICE: select a sub-cube using conditions on multiple dimensions
print("\n--- DICE: Region in (West, South) AND Category in (Electronics, Furniture) AND Year=2024 ---")
dice = cube_source[
    (cube_source["region"].isin(["West", "South"])) &
    (cube_source["category"].isin(["Electronics", "Furniture"])) &
    (cube_source["year"] == 2024)
].groupby(["region", "category"])[["net_sales", "profit"]].sum().round(0)
print(dice)

# ---------- PIVOT (ROTATE): swap axes for an alternate view of the same cube
print("\n--- PIVOT: rotate to Store (rows) x Category (columns) ---")
pivot_view = pd.pivot_table(
    cube_source, values="net_sales", index="store_name",
    columns="category", aggfunc="sum", fill_value=0
).round(0)
print(pivot_view)

# 6. EQUIVALENT SQL OLAP QUERIES ON THE DATA MART

print("\n" + "="*70)
print("STEP 6: SQL QUERIES AGAINST THE DATA MART")
print("="*70)

q_rollup = """
SELECT s.region, d.year, ROUND(SUM(f.net_sales),0) AS total_sales
FROM FactSales f
JOIN DimStore s ON f.store_key = s.store_key
JOIN DimDate  d ON f.date_key  = d.date_key
GROUP BY s.region, d.year
ORDER BY s.region, d.year;
"""
print("\n-- Roll-up: total sales by Region and Year --")
print(pd.read_sql(q_rollup, conn).to_string(index=False))

q_topcat = """
SELECT p.category, ROUND(SUM(f.net_sales),0) AS total_sales,
       ROUND(SUM(f.profit),0) AS total_profit
FROM FactSales f
JOIN DimProduct p ON f.product_key = p.product_key
GROUP BY p.category
ORDER BY total_sales DESC;
"""
print("\n-- Query: Total sales & profit by Category (ranked) --")
print(pd.read_sql(q_topcat, conn).to_string(index=False))

q_dice = """
SELECT s.region, p.category, d.quarter, ROUND(SUM(f.net_sales),0) AS sales
FROM FactSales f
JOIN DimStore s   ON f.store_key = s.store_key
JOIN DimProduct p ON f.product_key = p.product_key
JOIN DimDate d    ON f.date_key = d.date_key
WHERE s.region IN ('West','South')
  AND p.category IN ('Electronics','Furniture')
  AND d.year = 2024
GROUP BY s.region, p.category, d.quarter
ORDER BY s.region, p.category, d.quarter;
"""
print("\n-- Dice: West/South regions, Electronics/Furniture, Year 2024, by Quarter --")
print(pd.read_sql(q_dice, conn).to_string(index=False))

conn.close()
print("\nAll steps complete. Data mart saved at retail_datamart.db")

STEP 1: EXTRACT
Extracted 2000 raw transaction records
 txn_id   txn_date product_code product_name       category store_code      store_name      city     state region customer_id   segment  quantity  unit_price  discount_pct
T000000 2024-09-27         P007      Blender Home & Kitchen        S04         MG Road Bengaluru Karnataka  South       C0103 Wholesale         3       86.71            10
T000001 2025-04-03         P003   Headphones    Electronics        S03 Connaught Place     Delhi     Delhi  North       C0150    Retail         5      515.17             0
T000002 2025-02-17         P002   Smartphone    Electronics        S04         MG Road Bengaluru Karnataka  South       C0161 Wholesale         2      452.27            10

STEP 2: TRANSFORM (build star schema)
DimDate:     685 rows
DimProduct:  10 rows
DimStore:    5 rows
DimCustomer: 975 rows
FactSales:   2000 rows

STEP 3: LOAD (into SQLite data mart)
Loaded star schema into retail_datamart.db
Tables: ['DimDate', 'DimProdu

In [ ]:
import os
os.listdir()

['.config', 'retail_datamart.db', 'sample_data']

In [ ]:
from google.colab import files
files.download('retail_datamart.db')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import sqlite3, pandas as pd
conn = sqlite3.connect("retail_datamart.db")
pd.read_sql("SELECT * FROM DimProduct", conn)

,product_code,product_name,category,product_key
0,P007,Blender,Home & Kitchen,1
1,P003,Headphones,Electronics,2
2,P002,Smartphone,Electronics,3
3,P006,Jacket,Apparel,4
4,P005,Jeans,Apparel,5
5,P010,Office Chair,Furniture,6
6,P008,Microwave,Home & Kitchen,7
7,P004,T-Shirt,Apparel,8
8,P001,Laptop,Electronics,9
9,P009,Sofa,Furniture,10
